<a href="https://colab.research.google.com/github/guitorte/audio/blob/claude/music-stems-midi-workflow-04oJv/music-to-midi/notebooks/Song_to_Stems_to_MIDI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵→🎼 Music → Stems → MIDI (workflow único)

Um processo só: pega **uma música inteira**, separa em stems com **Demucs** e
transcreve cada stem para **MIDI multi-track**, salvando tudo organizado.
Unifica os dois cadernos antigos (`Stems_4` de separação + `Stem_to_MIDI_Batch`
de transcrição) numa única passagem `song_to_midi()`.

```
song.mp3
  └─► Demucs htdemucs_6s ─┬─► vocals  ─► Basic Pitch ──► vocals.mid
                          ├─► drums   ─► ADTOF ───────► drums.mid
                          ├─► bass    ─► Basic Pitch ──► bass.mid
                          ├─► guitar  ─► Basic Pitch ──► guitar.mid
                          ├─► piano   ─► Basic Pitch ──► piano.mid
                          └─► other   ─► Basic Pitch ──► other.mid
                                                            │
                                                            ▼
                                              merge → <track>.mid (multi-track)
```

**Saída organizada** em `OUTPUT_ROOT/<track>/`:

```
<track>/
├── stems/        vocals.wav, drums.wav, bass.wav, guitar.wav, piano.wav, other.wav
├── midi/         vocals.mid, drums.mid, ...   (um .mid por stem, p/ editar na DAW)
└── <track>.mid   MIDI multi-track consolidado
```

| Stem | Engine | GM program |
|---|---|---|
| `vocals` | Basic Pitch (ONNX) | 53 — Voice Oohs |
| `bass`   | Basic Pitch (ONNX) | 33 — Electric Bass |
| `guitar` | Basic Pitch (ONNX) | 27 — Electric Guitar (clean) |
| `piano`  | Basic Pitch (ONNX) |  0 — Acoustic Grand Piano |
| `other`  | Basic Pitch (ONNX) |  0 — Acoustic Grand Piano |
| `drums`  | ADTOF-pytorch      | canal 10 (GM drum map) |

## 1. Instalar dependências

Demucs (separação) + Basic Pitch via backend **ONNX** (transcrição pitched) +
ADTOF-pytorch (bateria).

⚠️ Após rodar esta célula, **reinicie o runtime** (`Runtime ▸ Restart Session`)
e siga das células seguintes.

In [ ]:
# Colab roda Python 3.12. basic-pitch 0.4.0 puxa tensorflow<2.15.1 /
# tflite-runtime, que não têm wheel 3.12. Workaround: instalar
# basic-pitch --no-deps e cair no backend ONNX (modelo nmp.onnx já embutido).
# Ver spotify/basic-pitch#188. ADTOF-pytorch é PyTorch-only (pesos embutidos).

!pip -q uninstall -y basic-pitch tensorflow tflite-runtime 2>/dev/null
!pip -q install demucs
!pip -q install basic-pitch --no-deps onnxruntime
!pip -q install "resampy<0.4.3" librosa pretty_midi mir_eval scikit-learn scipy typing_extensions soundfile mido matplotlib flatbuffers protobuf noisereduce
!pip -q install git+https://github.com/xavriley/ADTOF-pytorch.git

import importlib.util
_required = ('demucs', 'basic_pitch', 'onnxruntime', 'adtof_pytorch',
             'pretty_midi', 'librosa', 'soundfile', 'mido')
_missing = [m for m in _required if importlib.util.find_spec(m) is None]
print('=' * 60)
if _missing:
    print(f'Módulos não encontrados após install: {_missing}')
    print('   Rode a célula novamente; o pip às vezes precisa de 2 passadas.')
else:
    print('Instalação OK — Demucs + Basic Pitch (ONNX) + ADTOF-pytorch.')
print('Reinicie o runtime (Runtime > Restart Session) e siga adiante.')
print('=' * 60)

## 2. Imports e clone do repositório

Busca os módulos `music-to-midi/modules/` (onde mora `song_to_midi`). Em Colab
fazemos um shallow clone do repositório para `/content/audio`.

In [ ]:
import os, sys, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BRANCH = 'claude/music-stems-midi-workflow-04oJv'
if IN_COLAB:
    REPO_DIR = '/content/audio'
    if not os.path.isdir(REPO_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                               'https://github.com/guitorte/audio.git', REPO_DIR])
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

PKG_DIR = os.path.join(REPO_DIR, 'music-to-midi')
if PKG_DIR not in sys.path:
    sys.path.insert(0, PKG_DIR)

from modules import song_to_midi, DEMUCS_MODELS, DEFAULT_STEM_PROGRAMS  # noqa: E402
from IPython.display import Audio, display, FileLink  # noqa: E402

print(f'Colab          : {IN_COLAB}')
print(f'Pacote         : {PKG_DIR}')
print(f'Modelos Demucs : {list(DEMUCS_MODELS)}')

## 3. Montar Google Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Drive montado.')
else:
    print('Fora do Colab — pulando mount.')

## 4. Configurar parâmetros

Aponte `SONG_PATH` para a música. `htdemucs_6s` separa em 6 stems
(vocals/drums/bass/guitar/piano/other); os outros modelos dão 4 stems.

- `SHIFTS` / `OVERLAP`: qualidade vs. tempo da separação (notebook antigo usava 5 / 0.75).
- `STEM_RENAMES`: remapeia nomes de stem não-padrão p/ tipos conhecidos.
- `FORCE_SEPARATION`: re-roda o Demucs mesmo se já houver stems da última vez.

In [ ]:
# @title Parâmetros
SONG_PATH        = '/content/drive/MyDrive/music-to-midi/input/minha-musica.mp3'  # @param {type:'string'}
OUTPUT_ROOT      = '/content/drive/MyDrive/music-to-midi/output'  # @param {type:'string'}
MODEL            = 'htdemucs_6s'  # @param ['htdemucs_6s', 'htdemucs', 'htdemucs_ft', 'mdx_extra']
SHIFTS           = 1     # @param {type:'integer'}
OVERLAP          = 0.25  # @param {type:'number'}
FORCE_SEPARATION = False # @param {type:'boolean'}

# Remapeia stems com nome não-padrão. Ex.: {'Kim_Vocal_2': 'vocals'}
STEM_RENAMES = {}

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f'Música     : {SONG_PATH}')
print(f'Existe?    : {os.path.exists(SONG_PATH)}')
print(f'Saída em   : {OUTPUT_ROOT}')
print(f'Modelo     : {MODEL}')
if not os.path.exists(SONG_PATH):
    print('\n⚠️  Arquivo não encontrado — suba a música para o caminho acima.')

## 5. Rodar o workflow completo

Separa em stems e transcreve tudo numa passagem. Stems já separados de uma rodada
anterior são reaproveitados (a menos que `FORCE_SEPARATION=True`).

In [ ]:
import time
_t0 = time.time()
result = song_to_midi(
    SONG_PATH,
    OUTPUT_ROOT,
    model=MODEL,
    shifts=SHIFTS,
    overlap=OVERLAP,
    stem_types=STEM_RENAMES or None,
    skip_separation_if_exists=not FORCE_SEPARATION,
)
_dt = time.time() - _t0
tr = result.transcription

print('\n=== Resumo ===')
print(f'Faixa             : {result.track_name}')
print(f'Pasta             : {result.track_dir}')
print(f'Tempo total       : {_dt:.1f}s')
print(f'Stems transcritos : {len(tr.stem_results)}')
print(f'Total de notas    : {tr.total_notes}')
print(f'Duração do MIDI   : {tr.duration_s:.1f}s')
print(f'MIDI consolidado  : {result.merged_midi}')
print()
print(f'{"Stem":10s} {"Engine":14s} {"Notas":>6s} {"Duração":>10s}')
print('-' * 44)
for stem_type, r in tr.stem_results.items():
    print(f'{stem_type:10s} {r.engine:14s} {r.n_notes:6d} {r.duration_s:9.1f}s')

## 6. Piano roll multi-track

Cada track em uma cor. Drums (canal 10) ficam numa faixa separada com rótulos GM.

In [ ]:
merged = tr.midi_data
if not merged.instruments:
    print('Nenhuma faixa no MIDI.')
else:
    GM_DRUMS_SHORT = {35:'BD', 36:'Kick', 38:'Snare', 39:'Clap', 40:'ElSnr',
                      41:'LoFlT', 42:'HH', 43:'HiFlT', 44:'PdHH', 45:'LoT',
                      46:'OpHH', 47:'LMT', 48:'HMT', 49:'Crash', 50:'HiT', 51:'Ride'}
    COLORS = {'vocals':'#d62728', 'bass':'#1f77b4', 'guitar':'#2ca02c',
              'piano':'#ff7f0e', 'other':'#9467bd', 'drums':'#7f7f7f'}
    pitched = [i for i in merged.instruments if not i.is_drum and i.notes]
    drums   = [i for i in merged.instruments if i.is_drum and i.notes]
    n_rows  = (1 if pitched else 0) + (1 if drums else 0)
    if n_rows == 0:
        print('Nenhuma nota nos stems.')
    else:
        fig, axes = plt.subplots(n_rows, 1, figsize=(14, 4*n_rows), squeeze=False)
        ax_idx = 0
        if pitched:
            ax = axes[ax_idx, 0]; ax_idx += 1
            for inst in pitched:
                color = COLORS.get(inst.name, '#000000')
                for n in inst.notes:
                    ax.barh(n.pitch, n.end - n.start, left=n.start, height=0.9,
                            color=color, alpha=0.65, edgecolor='none')
                ax.scatter([], [], color=color, label=f'{inst.name} (n={len(inst.notes)})')
            ax.set_xlabel('Tempo (s)'); ax.set_ylabel('Pitch MIDI')
            ax.set_title('Pitched tracks', fontweight='bold')
            ax.legend(loc='upper right', framealpha=0.9)
            ax.grid(True, axis='y', alpha=0.3)
        if drums:
            ax = axes[ax_idx, 0]
            for inst in drums:
                for n in inst.notes:
                    ax.barh(n.pitch, max(0.05, n.end - n.start), left=n.start, height=0.85,
                            color=COLORS['drums'], alpha=0.8, edgecolor='none')
            unique_pitches = sorted({n.pitch for inst in drums for n in inst.notes})
            ax.set_yticks(unique_pitches)
            ax.set_yticklabels([GM_DRUMS_SHORT.get(p, f'p{p}') for p in unique_pitches])
            ax.set_xlabel('Tempo (s)')
            ax.set_title('Drum track (GM channel 10)', fontweight='bold')
            ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout(); plt.show()

## 7. Preview sonoro do MIDI consolidado

Pitched: senoidal via `pretty_midi`. Drums: bursts de ruído/tom por hit.

In [ ]:
sr_preview = 22050
dur_total = max(tr.duration_s, 0.5)
wave = np.zeros(int(dur_total * sr_preview) + sr_preview, dtype=np.float32)
for inst in merged.instruments:
    if not inst.notes:
        continue
    if inst.is_drum:
        for nt in inst.notes:
            idx = int(nt.start * sr_preview)
            length = int(0.08 * sr_preview)
            env = np.exp(-np.linspace(0, 6, length))
            t = np.arange(length) / sr_preview
            if nt.pitch in (35, 36):
                tone = np.sin(2*np.pi*60*t)
            elif nt.pitch in (38, 40):
                tone = np.random.randn(length) * 0.5 + 0.3 * np.sin(2*np.pi*200*t)
            else:
                tone = np.random.randn(length) * 0.8
            sl = wave[idx:idx+length]
            sl[:] = sl + (tone * env * (nt.velocity/127.0)).astype(np.float32)[:len(sl)]
    else:
        chunk = inst.synthesize(fs=sr_preview)
        if len(chunk) > len(wave):
            chunk = chunk[:len(wave)]
        wave[:len(chunk)] += chunk.astype(np.float32)
peak = float(np.max(np.abs(wave)))
if peak > 0:
    wave = (0.9 * wave / peak).astype(np.float32)
print('MIDI consolidado (preview sintetizado):')
display(Audio(wave, rate=sr_preview))

## 8. Arquivos gerados

O consolidado fica em `<track>/<track>.mid`; os per-stem `.mid` em `<track>/midi/`
(úteis para editar isoladamente na DAW); os stems de áudio em `<track>/stems/`.

In [ ]:
print(f'MIDI consolidado : {result.merged_midi}')
display(FileLink(result.merged_midi))
print('\nPer-stem .mid:')
for stem_type in tr.stem_results:
    p = os.path.join(result.midi_dir, f'{stem_type}.mid')
    if os.path.exists(p):
        print(f'  {stem_type:8s} -> {p}')
        display(FileLink(p))
print('\nStems de áudio:')
for stem_type, p in result.separation.stems.items():
    print(f'  {stem_type:8s} -> {p}')
if IN_COLAB:
    try:
        from google.colab import files
        files.download(result.merged_midi)
    except Exception as e:
        print(f'(download manual via Drive — botão automático falhou: {e})')